In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_28.pickle

In [3]:
%%RecordEvent
%%time
### cell 28 ###

def grab_subset_of_data_40(original_df, question_of_interest):
    """
    Vectorized subset of columns containing the question text, rename to the suffix after '-' and drop rows
    where all answers are NaN.
    """
    return (
        original_df
        .filter(like=question_of_interest)
        .rename(columns=lambda col: col.rsplit('-', 1)[-1].lstrip())
        .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest,
        include_2017=False
    ):
    """
    For each year (optionally including 2017), grab the subset for the question, tag with year,
    concatenate, and return both the combined DataFrame and the per-year counts.
    """
    sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        sources.insert(0, ('2017', responses_df_2017))

    dfs = []
    for year, df_src in sources:
        df_sub = grab_subset_of_data_40(df_src, question_of_interest)
        if not df_sub.empty:
            dfs.append(df_sub.assign(year=year))

    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined_counts = df_combined.groupby('year', sort=True).count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_40(df, df_counts):
    """
    Given the full concatenated DataFrame df (with a 'year' column) and its per-year counts,
    convert those counts to percentages of respondents per year.
    """
    # Total respondents in each year
    totals = df['year'].value_counts().sort_index()

    # Divide each count by the total respondents for that year and multiply by 100
    df_percent = (
        df_counts
        .set_index('year')
        .div(totals, axis=0)
        .mul(100)
        .reset_index()
    )
    return df_percent


# Standardize the question text in the 2020 and 2021 column names
q_orig = (
    'Which of the following hosted notebook products do you use on a regular basis?'
)
q_new = (
    'Do you use any of the following hosted notebook products?'
)
for df in (responses_df_2020, responses_df_2021):
    df.columns = df.columns.str.replace(q_orig, q_new, regex=False)

# Bulk rename and replace answers in the 2019 and 2018 DataFrames
replacements_2019 = {
    'Google Colab ': 'Colab Notebooks',
    'Kaggle Notebooks (Kernels) ': 'Kaggle Notebooks'
}
for old, new in replacements_2019.items():
    responses_df_2019_cell10.columns = (
        responses_df_2019_cell10.columns.str.replace(old, new, regex=False)
    )
    responses_df_2019_cell10.replace(old, new, regex=False, inplace=True)

replacements_2018 = {
    'Google Colab': 'Colab Notebooks',
    'Kaggle Kernels': 'Kaggle Notebooks'
}
for old, new in replacements_2018.items():
    responses_df_2018_cell10.columns = (
        responses_df_2018_cell10.columns.str.replace(old, new, regex=False)
    )
    responses_df_2018_cell10.replace(old, new, regex=False, inplace=True)

# Combine responses, count, convert to percentages, and reshape for plotting
question_of_interest_cell40 = (
    'Do you use any of the following hosted notebook products?'
)
notebooks_df_combined, notebooks_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined,
    notebooks_df_combined_counts
)

df_cell40 = (
    notebooks_df_combined_percentages
    .loc[:, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']]
    .melt(
        id_vars='year',
        value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks']
    )
    .rename(columns={'variable': ''})
)

df_cell40.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    9 non-null      object 
 1           9 non-null      object 
 2   value   9 non-null      float64
dtypes: float64(1), object(2)
memory usage: 344.0+ bytes
CPU times: user 1.08 s, sys: 7.55 ms, total: 1.08 s
Wall time: 1.08 s


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_28_try_2.pickle